In [ ]:
# Cell 1: Install all required libraries
!pip install transformers sentence-transformers PyPDF2 torch -q

print("✅ All libraries installed successfully!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 12.4 MB/s eta 0:00:00
✅ All libraries installed successfully!


In [ ]:
# Cell 2: Import all necessary modules
import re
import textwrap
from transformers import T5ForConditionalGeneration, T5Tokenizer
from sentence_transformers import SentenceTransformer, util
import PyPDF2
from google.colab import files
import io

print("✅ Imports done!")

✅ Imports done!


In [ ]:
# Cell 3: Load the free open-source AI models from Hugging Face
# This may take a minute or two on first run

print("⏳ Loading FLAN-T5 model for rewriting...")
flan_tokenizer = T5Tokenizer.from_pretrained("google/flan-t5-base")
flan_model = T5ForConditionalGeneration.from_pretrained("google/flan-t5-base")
print("✅ FLAN-T5 loaded!")

print("⏳ Loading MiniLM model for similarity scoring...")
minilm_model = SentenceTransformer("all-MiniLM-L6-v2")
print("✅ MiniLM loaded!")

print("\n🎉 All models ready!")

⏳ Loading FLAN-T5 model for rewriting...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ MiniLM loaded!

🎉 All models ready!


In [ ]:
# Cell 4: Choose how to provide your resume

print("=" * 55)
print("       AI RESUME REWRITER — INPUT METHOD")
print("=" * 55)
print("\nHow would you like to provide your resume?")
print("  1 → Upload a PDF file")
print("  2 → Paste text directly")
print("-" * 55)

choice = input("Enter 1 or 2: ").strip()

resume_text = ""

# ── OPTION 1: Upload PDF ──────────────────────────────────
if choice == "1":
    print("\n📂 Please upload your resume PDF file...")
    uploaded = files.upload()

    for filename, content in uploaded.items():
        pdf_reader = PyPDF2.PdfReader(io.BytesIO(content))
        for page in pdf_reader.pages:
            resume_text += page.extract_text() + "\n"

    print(f"\n✅ PDF '{filename}' loaded successfully!")
    print(f"📝 Extracted {len(resume_text)} characters.")

# ── OPTION 2: Paste Text ──────────────────────────────────
elif choice == "2":
    print("\n📋 Paste your resume text below.")
    print("When done, type END on a new line and press Enter:\n")
    lines = []
    while True:
        line = input()
        if line.strip().upper() == "END":
            break
        lines.append(line)
    resume_text = "\n".join(lines)
    print(f"\n✅ Resume text received! ({len(resume_text)} characters)")

else:
    print("❌ Invalid choice. Please re-run this cell and enter 1 or 2.")

# Preview
print("\n📄 Preview (first 300 characters):")
print("-" * 55)
print(resume_text[:300] + "...")

       AI RESUME REWRITER — INPUT METHOD

How would you like to provide your resume?
  1 → Upload a PDF file
  2 → Paste text directly
-------------------------------------------------------
Enter 1 or 2: 2

📋 Paste your resume text below.
When done, type END on a new line and press Enter:

NAME: Darshini M S  EMAIL: [darshini@example.com](mailto:darshini@example.com) PHONE: 9876543210  OBJECTIVE: Motivated Computer Science student seeking an entry-level role in AI and software development to apply problem-solving skills and contribute to innovative projects.  EDUCATION: Bachelor of Engineering in Computer Science XYZ Engineering College CGPA: 8.7  SKILLS: Python, Java, SQL, HTML, CSS, JavaScript Basic Machine Learning Data Structures and Algorithms Problem Solving  PROJECTS:  1. Chatbot Application    Developed a rule-based chatbot using Python to answer user queries. Implemented basic NLP techniques like tokenization.  2. Student Result Management System    Created a web-based system

In [ ]:
# Cell 5: Enter your target job role and optional job description

print("=" * 55)
print("          ENTER TARGET ROLE & JOB DESCRIPTION")
print("=" * 55)

print("\nCommon roles: AI Engineer, Software Developer,")
print("Data Analyst, Web Developer, Machine Learning Engineer\n")

job_role = input("🎯 Enter your target job role: ").strip()

print("\n📋 Paste the Job Description (optional).")
print("Press ENTER then type END to skip or finish:\n")

jd_lines = []
while True:
    line = input()
    if line.strip().upper() == "END" or line.strip() == "":
        break
    jd_lines.append(line)

job_description = " ".join(jd_lines).strip()

if not job_description:
    # Auto-generate a basic JD based on role if user skips
    job_description = f"We are looking for a skilled {job_role} with strong communication, problem-solving, and technical skills. Experience with projects, teamwork, and delivering results is preferred."

print(f"\n✅ Role: {job_role}")
print(f"✅ Job Description: {job_description[:100]}...")

          ENTER TARGET ROLE & JOB DESCRIPTION

Common roles: AI Engineer, Software Developer,
Data Analyst, Web Developer, Machine Learning Engineer

🎯 Enter your target job role: AI Engineer

📋 Paste the Job Description (optional).
Press ENTER then type END to skip or finish:

END

✅ Role: AI Engineer
✅ Job Description: We are looking for a skilled AI Engineer with strong communication, problem-solving, and technical s...


In [ ]:
# Cell 6: Split resume into sections using simple rule-based approach

def split_resume_sections(text):
    """
    Splits resume text into: summary, skills, projects, experience, education
    Uses keyword matching — no ML needed here.
    """
    sections = {
        "summary":    "",
        "skills":     "",
        "projects":   "",
        "experience": "",
        "education":  ""
    }

    # Lowercase copy for matching keywords
    lower = text.lower()

    # Define section keyword markers
    markers = {
        "summary":    ["summary", "objective", "profile", "about me"],
        "skills":     ["skills", "technical skills", "core competencies", "technologies"],
        "projects":   ["projects", "project work", "key projects"],
        "experience": ["experience", "work experience", "employment", "internship"],
        "education":  ["education", "academic", "qualification"]
    }

    # Find positions of each section header in the text
    positions = {}
    for section, keywords in markers.items():
        for kw in keywords:
            idx = lower.find(kw)
            if idx != -1:
                positions[section] = idx
                break

    # Sort sections by their position in the resume
    sorted_sections = sorted(positions.items(), key=lambda x: x[1])

    # Extract text between section headers
    for i, (section, start) in enumerate(sorted_sections):
        end = sorted_sections[i + 1][1] if i + 1 < len(sorted_sections) else len(text)
        sections[section] = text[start:end].strip()

    # If no sections found at all, put everything in summary
    if all(v == "" for v in sections.values()):
        sections["summary"] = text.strip()

    return sections

# Run the splitter
resume_sections = split_resume_sections(resume_text)

print("=" * 55)
print("       RESUME SECTIONS DETECTED")
print("=" * 55)
for section, content in resume_sections.items():
    status = "✅" if content else "⬜"
    print(f"  {status} {section.upper()} — {len(content)} characters")

       RESUME SECTIONS DETECTED
  ✅ SUMMARY — 129 characters
  ✅ SKILLS — 35 characters
  ✅ PROJECTS — 9 characters
  ✅ EXPERIENCE — 381 characters
  ✅ EDUCATION — 634 characters


In [ ]:
# Cell 7: Rewrite each section using FLAN-T5

def rewrite_section(section_name, section_text, job_role):
    """
    Uses FLAN-T5 to rewrite a resume section professionally.
    """
    if not section_text.strip():
        return f"[No {section_name} section found in your resume]"

    # Truncate long sections so they fit in model input
    section_text = section_text[:400]

    # Build a clear instruction prompt for FLAN-T5
    prompt = (
        f"Rewrite the following resume {section_name} section to be more professional, "
        f"impactful, and ATS-friendly for a {job_role} role. "
        f"Use strong action verbs, add measurable results where possible, "
        f"and highlight relevant skills.\n\n"
        f"Original:\n{section_text}\n\n"
        f"Improved version:"
    )

    # Tokenize the prompt
    inputs = flan_tokenizer(
        prompt,
        return_tensors="pt",
        max_length=512,
        truncation=True
    )

    # Generate the rewritten text
    outputs = flan_model.generate(
        inputs["input_ids"],
        max_new_tokens=200,
        num_beams=4,
        early_stopping=True
    )

    # Decode and return
    result = flan_tokenizer.decode(outputs[0], skip_special_tokens=True)
    return result.strip()


print("=" * 55)
print("     AI REWRITING YOUR RESUME — PLEASE WAIT...")
print("=" * 55)

rewritten_sections = {}

for section, content in resume_sections.items():
    if content:
        print(f"\n✍️  Rewriting {section.upper()} section...")
        rewritten_sections[section] = rewrite_section(section, content, job_role)
        print(f"   ✅ Done!")
    else:
        rewritten_sections[section] = ""

print("\n🎉 All sections rewritten successfully!")

     AI REWRITING YOUR RESUME — PLEASE WAIT...

✍️  Rewriting SUMMARY section...
   ✅ Done!

✍️  Rewriting SKILLS section...
   ✅ Done!

✍️  Rewriting PROJECTS section...
   ✅ Done!

✍️  Rewriting EXPERIENCE section...
   ✅ Done!

✍️  Rewriting EDUCATION section...
   ✅ Done!

🎉 All sections rewritten successfully!


In [ ]:
# Cell 8: Score resume against job description using MiniLM

def calculate_similarity(resume_text, job_description):
    """
    Compares resume text with job description using MiniLM sentence embeddings.
    Returns a similarity score from 0 to 100.
    """
    resume_embedding = minilm_model.encode(resume_text, convert_to_tensor=True)
    jd_embedding     = minilm_model.encode(job_description, convert_to_tensor=True)

    # Cosine similarity (value between -1 and 1, we scale to 0–100)
    score = util.cos_sim(resume_embedding, jd_embedding).item()
    return round(score * 100, 2)


# Combine all rewritten sections for scoring
full_rewritten_resume = " ".join(
    [v for v in rewritten_sections.values() if v]
)

score = calculate_similarity(full_rewritten_resume, job_description)

print("=" * 55)
print("          RESUME MATCH SCORE")
print("=" * 55)

bar_filled = int(score / 5)  # 20 blocks total
bar = "█" * bar_filled + "░" * (20 - bar_filled)
print(f"\n  [{bar}]  {score}%\n")

if score >= 80:
    print("  🏆 Excellent match! Your resume aligns very well.")
elif score >= 60:
    print("  ✅ Good match. Minor tweaks can improve your score.")
elif score >= 40:
    print("  ⚠️  Average match. Consider adding more relevant keywords.")
else:
    print("  ❌ Low match. Your resume needs significant improvements.")

          RESUME MATCH SCORE

  [███████████░░░░░░░░░]  57.21%

  ⚠️  Average match. Consider adding more relevant keywords.


In [ ]:
# Cell 9: Suggest improvements if score is below 60%

def extract_keywords(text, top_n=15):
    """
    Simple keyword extractor — finds important words (no ML needed).
    Filters out common stop words.
    """
    stop_words = {
        "and", "the", "to", "of", "in", "a", "is", "for", "with",
        "we", "are", "you", "on", "at", "be", "will", "have", "this",
        "that", "an", "as", "by", "or", "from", "it", "our", "your"
    }
    words = re.findall(r'\b[a-zA-Z]{4,}\b', text.lower())
    freq = {}
    for word in words:
        if word not in stop_words:
            freq[word] = freq.get(word, 0) + 1
    sorted_words = sorted(freq, key=freq.get, reverse=True)
    return sorted_words[:top_n]


print("=" * 55)
print("         SUGGESTIONS & IMPROVEMENTS")
print("=" * 55)

if score < 60:
    jd_keywords     = set(extract_keywords(job_description))
    resume_keywords = set(extract_keywords(full_rewritten_resume))

    missing = jd_keywords - resume_keywords

    print(f"\n⚠️  Score is {score}% — here's how to improve:\n")

    print("  📌 Missing Keywords from Job Description:")
    if missing:
        for kw in list(missing)[:10]:
            print(f"     → {kw}")
    else:
        print("     → No critical keywords missing!")

    print("\n  💡 General Suggestions:")
    print("     → Add measurable achievements (e.g., 'improved accuracy by 20%')")
    print("     → Use action verbs: developed, built, optimized, deployed")
    print("     → Include role-specific tools and technologies")
    print(f"     → Tailor your resume more specifically to '{job_role}'")
else:
    print(f"\n✅ Score is {score}% — your resume is well-matched!")
    print("   A few tips to make it even stronger:")
    print("   → Quantify achievements wherever possible")
    print("   → Ensure formatting is clean for ATS systems")

         SUGGESTIONS & IMPROVEMENTS

⚠️  Score is 57.21% — here's how to improve:

  📌 Missing Keywords from Job Description:
     → technical
     → strong
     → engineer
     → skilled
     → teamwork
     → communication
     → skills
     → projects
     → delivering
     → results

  💡 General Suggestions:
     → Add measurable achievements (e.g., 'improved accuracy by 20%')
     → Use action verbs: developed, built, optimized, deployed
     → Include role-specific tools and technologies
     → Tailor your resume more specifically to 'AI Engineer'


In [ ]:
# Cell 10: Simple resume chatbot for interactive improvements

# Chat memory — stores context across conversation
chat_context = {
    "job_role":          job_role,
    "resume_sections":   rewritten_sections,
    "job_description":   job_description,
    "score":             score,
    "history":           []
}


def chatbot_respond(user_message, context):
    """
    Simple chatbot that responds to resume-related questions.
    Uses FLAN-T5 for dynamic responses.
    """
    msg = user_message.lower().strip()

    # ── Rule-based responses ──────────────────────────────
    if "score" in msg or "match" in msg:
        return f"📊 Your current resume match score is {context['score']}%."

    if "skills" in msg and ("improve" in msg or "rewrite" in msg or "better" in msg):
        section = context["resume_sections"].get("skills", "")
        if section:
            improved = rewrite_section("skills", section, context["job_role"])
            context["resume_sections"]["skills"] = improved
            return f"✍️ Here's your improved Skills section:\n\n{improved}"
        return "⚠️ No skills section found in your resume."

    if "ats" in msg:
        return (
            "📋 ATS (Applicant Tracking System) Tips:\n"
            "  → Use standard section headers (Skills, Experience, Education)\n"
            "  → Avoid tables, images, or columns\n"
            "  → Include keywords from the job description\n"
            "  → Use simple bullet points with action verbs"
        )

    if "summary" in msg and ("improve" in msg or "rewrite" in msg):
        section = context["resume_sections"].get("summary", "")
        if section:
            improved = rewrite_section("summary", section, context["job_role"])
            context["resume_sections"]["summary"] = improved
            return f"✍️ Here's your improved Summary:\n\n{improved}"
        return "⚠️ No summary section found."

    if "project" in msg and ("improve" in msg or "rewrite" in msg):
        section = context["resume_sections"].get("projects", "")
        if section:
            improved = rewrite_section("projects", section, context["job_role"])
            context["resume_sections"]["projects"] = improved
            return f"✍️ Here's your improved Projects section:\n\n{improved}"
        return "⚠️ No projects section found."

    if "tip" in msg or "suggest" in msg or "advice" in msg:
        return (
            f"💡 Tips for {context['job_role']} roles:\n"
            "  → Highlight technical tools and frameworks\n"
            "  → Show impact with numbers (e.g., 'reduced time by 30%')\n"
            "  → Keep resume to 1 page if under 2 years experience\n"
            "  → Tailor each application to the specific job"
        )

    if "help" in msg:
        return (
            "🤖 I can help you with:\n"
            "  → 'Improve my skills section'\n"
            "  → 'Rewrite my summary'\n"
            "  → 'Make this ATS friendly'\n"
            "  → 'Show my score'\n"
            "  → 'Give me tips'\n"
            "  → 'Improve my projects section'"
        )

    # ── FLAN-T5 fallback for other questions ─────────────
    prompt = (
        f"You are an expert resume coach. The user is applying for a {context['job_role']} role. "
        f"Their resume score is {context['score']}%. "
        f"Answer this question helpfully: {user_message}"
    )
    inputs = flan_tokenizer(prompt, return_tensors="pt", max_length=256, truncation=True)
    outputs = flan_model.generate(inputs["input_ids"], max_new_tokens=150, num_beams=3)
    return flan_tokenizer.decode(outputs[0], skip_special_tokens=True)


# ── Start Chatbot Loop ────────────────────────────────────
print("=" * 55)
print("     🤖 AI RESUME CHATBOT — TYPE 'quit' TO EXIT")
print("=" * 55)
print("  Try asking:")
print("  → 'Improve my skills section'")
print("  → 'Make this ATS friendly'")
print("  → 'Show my score'")
print("  → 'Give me tips'")
print("-" * 55)

while True:
    user_input = input("\n👤 You: ").strip()

    if not user_input:
        continue
    if user_input.lower() in ["quit", "exit", "bye"]:
        print("🤖 Bot: Goodbye! Best of luck with your job applications! 🚀")
        break

    # Save to history
    chat_context["history"].append({"user": user_input})

    # Get response
    response = chatbot_respond(user_input, chat_context)
    chat_context["history"][-1]["bot"] = response

    print(f"\n🤖 Bot: {response}")

     🤖 AI RESUME CHATBOT — TYPE 'quit' TO EXIT
  Try asking:
  → 'Improve my skills section'
  → 'Make this ATS friendly'
  → 'Show my score'
  → 'Give me tips'
-------------------------------------------------------

👤 You: Make this ATS friendly

🤖 Bot: 📋 ATS (Applicant Tracking System) Tips:
  → Use standard section headers (Skills, Experience, Education)
  → Avoid tables, images, or columns
  → Include keywords from the job description
  → Use simple bullet points with action verbs

👤 You: Include keywords from the job description

🤖 Bot: AI, engineer, resume

👤 You: Machine Learning Deep Learning Natural Language Processing (NLP) Computer Vision Neural Networks Supervised Learning Unsupervised Learning Reinforcement Learning

🤖 Bot: Deep Reinforcing Learning Reinforcing Learning

👤 You: Deep Learning

🤖 Bot: Deep learning.


KeyboardInterrupt: Interrupted by user

In [ ]:
# Cell 11: Display the complete rewritten resume

print("=" * 55)
print("         ✨ YOUR REWRITTEN RESUME ✨")
print("=" * 55)

section_order = ["summary", "skills", "experience", "projects", "education"]

for section in section_order:
    content = rewritten_sections.get(section, "")
    if content:
        print(f"\n{'─' * 55}")
        print(f"  📌 {section.upper()}")
        print(f"{'─' * 55}")
        # Wrap long lines nicely
        for line in textwrap.wrap(content, width=70):
            print("  " + line)

print(f"\n{'=' * 55}")
print(f"  🎯 Target Role : {job_role}")
print(f"  📊 Match Score : {score}%")
print(f"{'=' * 55}")
print("\n✅ Resume rewriting complete! Good luck! 🚀")